<a href="https://colab.research.google.com/github/P-eter-shi/marketAnalyser/blob/main/AdStockStorage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import asyncio
import aiohttp
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import re
import json
import time
import logging
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, asdict
import hashlib
from urllib.parse import urljoin, urlparse
import schedule
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

# Enhanced NLP and Sentiment Analysis
from textblob import TextBlob
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer

# Advanced Web Scraping with Selenium
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.common.exceptions import TimeoutException, NoSuchElementException

# Data Processing and ML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from collections import Counter
import yfinance as yf

# Initialize NLTK components
try:
    nltk.download('vader_lexicon', quiet=True)
    nltk.download('punkt', quiet=True)
    nltk.download('stopwords', quiet=True)
    nltk.download('wordnet', quiet=True)
except:
    pass

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('kenyan_sentiment.log'),
        logging.StreamHandler()
    ]
)

@dataclass
class SentimentRecord:
    """Enhanced data class for sentiment records with Kenyan market context"""
    timestamp: datetime
    symbol: str
    company_name: str
    source: str
    content: str
    sentiment_score: float
    confidence: float
    category: str
    url: str = ""
    author: str = ""
    engagement_score: float = 0.0
    sector: str = ""
    market_cap_impact: float = 0.0
    news_type: str = ""  # earnings, regulatory, macro, sector, company
    language: str = "en"

class EnhancedDatabaseManager:
    """Enhanced database manager for Kenyan market sentiment data"""

    def __init__(self, db_path: str = "kenyan_market_sentiment.db"):
        self.db_path = db_path
        self.init_database()

    def init_database(self):
        """Initialize comprehensive database schema"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()

        # Main sentiment data table with Kenyan market specifics
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS sentiment_data (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                timestamp DATETIME,
                symbol TEXT,
                company_name TEXT,
                source TEXT,
                content TEXT,
                sentiment_score REAL,
                confidence REAL,
                category TEXT,
                url TEXT,
                author TEXT,
                engagement_score REAL,
                sector TEXT,
                market_cap_impact REAL,
                news_type TEXT,
                language TEXT,
                content_hash TEXT UNIQUE,
                created_at DATETIME DEFAULT CURRENT_TIMESTAMP
            )
        ''')

        # Enhanced aggregated scores table
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS aggregated_scores (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                timestamp DATETIME,
                symbol TEXT,
                company_name TEXT,
                period_start DATETIME,
                period_end DATETIME,
                average_sentiment REAL,
                positive_count INTEGER,
                negative_count INTEGER,
                neutral_count INTEGER,
                total_mentions INTEGER,
                confidence_weighted_score REAL,
                volatility_score REAL,
                trending_topics TEXT,
                sector_sentiment REAL,
                market_momentum REAL,
                regulatory_impact REAL,
                economic_context REAL,
                prediction_confidence REAL,
                created_at DATETIME DEFAULT CURRENT_TIMESTAMP
            )
        ''')

        # Kenyan companies reference table
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS kenyan_companies (
                symbol TEXT PRIMARY KEY,
                company_name TEXT,
                sector TEXT,
                market_cap REAL,
                listing_date DATE,
                website TEXT,
                description TEXT,
                is_active BOOLEAN DEFAULT TRUE,
                last_updated DATETIME DEFAULT CURRENT_TIMESTAMP
            )
        ''')

        # Data sources tracking with Kenyan focus
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS data_sources (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                source_name TEXT UNIQUE,
                base_url TEXT,
                source_type TEXT,
                reliability_score REAL,
                last_scraped DATETIME,
                success_rate REAL,
                avg_response_time REAL,
                is_active BOOLEAN DEFAULT TRUE,
                kenyan_focus BOOLEAN DEFAULT FALSE
            )
        ''')

        # Economic indicators table for context
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS economic_indicators (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                date DATE,
                indicator_name TEXT,
                value REAL,
                unit TEXT,
                source TEXT,
                impact_level INTEGER,
                created_at DATETIME DEFAULT CURRENT_TIMESTAMP
            )
        ''')

        # Initialize Kenyan companies data
        self.populate_kenyan_companies()
        self.populate_data_sources()

        conn.commit()
        conn.close()

    def populate_kenyan_companies(self):
        """Populate Kenyan companies reference data"""
        kenyan_companies = [
            # Banking Sector
            ('EQTY', 'Equity Group Holdings Plc', 'Banking', 'Large Cap'),
            ('KCB', 'KCB Group Plc', 'Banking', 'Large Cap'),
            ('COOP', 'Co-operative Bank of Kenya Ltd', 'Banking', 'Large Cap'),
            ('ABSA', 'Absa Bank Kenya Plc', 'Banking', 'Large Cap'),
            ('SCBK', 'Standard Chartered Bank Kenya Ltd', 'Banking', 'Large Cap'),
            ('DTK', 'Diamond Trust Bank Kenya Ltd', 'Banking', 'Mid Cap'),
            ('HF', 'HF Group Ltd', 'Banking', 'Small Cap'),

            # Telecommunications
            ('SCOM', 'Safaricom Plc', 'Telecommunications', 'Large Cap'),

            # Energy & Utilities
            ('KPLC', 'Kenya Power and Lighting Co Ltd', 'Utilities', 'Large Cap'),
            ('KEGN', 'KenGen Co Ltd', 'Utilities', 'Large Cap'),
            ('UMME', 'Umeme Ltd', 'Utilities', 'Mid Cap'),

            # Manufacturing
            ('EABL', 'East African Breweries Ltd', 'Manufacturing', 'Large Cap'),
            ('BAT', 'British American Tobacco Kenya Plc', 'Manufacturing', 'Large Cap'),
            ('CARB', 'Carbacid Investments Plc', 'Manufacturing', 'Mid Cap'),
            ('BOC', 'BOC Kenya Plc', 'Manufacturing', 'Mid Cap'),

            # Insurance
            ('BRIT', 'Britam Holdings Plc', 'Insurance', 'Large Cap'),
            ('JUBILEE', 'Jubilee Holdings Ltd', 'Insurance', 'Large Cap'),
            ('CIC', 'CIC Insurance Group Ltd', 'Insurance', 'Mid Cap'),
            ('LIBERTY', 'Liberty Kenya Holdings Ltd', 'Insurance', 'Mid Cap'),

            # Real Estate & Construction
            ('ARM', 'ARM Cement Ltd', 'Construction', 'Mid Cap'),
            ('BAMB', 'Bamburi Cement Ltd', 'Construction', 'Mid Cap'),

            # Agriculture & Food
            ('KAKUZI', 'Kakuzi Plc', 'Agriculture', 'Small Cap'),
            ('FLAME', 'Flame Tree Group Holdings Ltd', 'Agriculture', 'Small Cap'),

            # Commercial Services
            ('NMG', 'Nation Media Group Plc', 'Media', 'Mid Cap'),
            ('SCAN', 'Scangroup Plc', 'Commercial Services', 'Small Cap'),

            # Investment Services
            ('CIC', 'CIC Insurance Group Ltd', 'Investment Services', 'Mid Cap'),
            ('OLYMPIA', 'Olympia Capital Holdings Ltd', 'Investment Services', 'Small Cap'),
        ]

        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()

        for symbol, name, sector, market_cap in kenyan_companies:
            cursor.execute('''
                INSERT OR REPLACE INTO kenyan_companies
                (symbol, company_name, sector, market_cap)
                VALUES (?, ?, ?, ?)
            ''', (symbol, name, sector, market_cap))

        conn.commit()
        conn.close()

    def populate_data_sources(self):
        """Populate Kenyan-focused data sources"""
        data_sources = [
            ('business_daily', 'https://www.businessdailyafrica.com/', 'news', 0.9, True),
            ('standard_digital', 'https://www.standardmedia.co.ke/business/', 'news', 0.8, True),
            ('nation_media', 'https://nation.africa/kenya/business/', 'news', 0.85, True),
            ('capital_fm', 'https://www.capitalfm.co.ke/business/', 'news', 0.75, True),
            ('citizen_digital', 'https://www.citizen.digital/business/', 'news', 0.7, True),
            ('kenyan_wallstreet', 'https://kenyanwallstreet.com/', 'financial', 0.9, True),
            ('nse_kenya', 'https://www.nse.co.ke/', 'official', 0.95, True),
            ('cma_kenya', 'https://www.cma.or.ke/', 'regulatory', 0.95, True),
            ('cbk_kenya', 'https://www.centralbank.go.ke/', 'regulatory', 0.95, True),
            ('reuters_kenya', 'https://www.reuters.com/world/africa/kenya/', 'international', 0.9, False),
            ('bloomberg_africa', 'https://www.bloomberg.com/africa/', 'international', 0.9, False),
            ('african_business', 'https://african.business/markets/', 'regional', 0.8, False),
        ]

        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()

        for name, url, source_type, reliability, kenyan_focus in data_sources:
            cursor.execute('''
                INSERT OR REPLACE INTO data_sources
                (source_name, base_url, source_type, reliability_score, kenyan_focus)
                VALUES (?, ?, ?, ?, ?)
            ''', (name, url, source_type, reliability, kenyan_focus))

        conn.commit()
        conn.close()

    def insert_sentiment(self, record: SentimentRecord) -> bool:
        """Insert sentiment record with enhanced error handling"""
        try:
            conn = sqlite3.connect(self.db_path)
            cursor = conn.cursor()

            # Create content hash to avoid duplicates
            content_hash = hashlib.md5(f"{record.content}{record.source}".encode()).hexdigest()

            cursor.execute('''
                INSERT OR IGNORE INTO sentiment_data
                (timestamp, symbol, company_name, source, content, sentiment_score,
                 confidence, category, url, author, engagement_score, sector,
                 market_cap_impact, news_type, language, content_hash)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ''', (
                record.timestamp, record.symbol, record.company_name, record.source,
                record.content, record.sentiment_score, record.confidence, record.category,
                record.url, record.author, record.engagement_score, record.sector,
                record.market_cap_impact, record.news_type, record.language, content_hash
            ))

            conn.commit()
            conn.close()
            return True

        except Exception as e:
            logging.error(f"Database insert error: {e}")
            return False

    def get_company_info(self, symbol: str) -> Dict:
        """Get company information from reference table"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()

        cursor.execute('''
            SELECT * FROM kenyan_companies WHERE symbol = ?
        ''', (symbol,))

        result = cursor.fetchone()
        conn.close()

        if result:
            columns = ['symbol', 'company_name', 'sector', 'market_cap', 'listing_date',
                      'website', 'description', 'is_active', 'last_updated']
            return dict(zip(columns, result))

        return {'symbol': symbol, 'company_name': symbol, 'sector': 'Unknown', 'market_cap': 'Unknown'}

class KenyanSentimentAnalyzer:
    """Advanced sentiment analyzer optimized for Kenyan market context"""

    def __init__(self):
        self.sia = SentimentIntensityAnalyzer()
        self.lemmatizer = WordNetLemmatizer()
        self.stop_words = set(stopwords.words('english'))

        # Kenyan market-specific keywords
        self.kenyan_financial_keywords = {
            'positive': [
                # Market Performance
                'growth', 'profit', 'dividend', 'expansion', 'acquisition', 'merger',
                'ipo', 'listing', 'rally', 'bullish', 'surge', 'uptick', 'rebound',
                'recovery', 'breakthrough', 'milestone', 'record high', 'all-time high',

                # Banking & Finance (Kenya specific)
                'loan growth', 'deposit growth', 'net interest margin', 'npl reduction',
                'digital banking', 'mobile banking', 'agent banking', 'diaspora remittances',
                'financial inclusion', 'credit rating upgrade', 'capital adequacy',

                # Telecom (Safaricom focus)
                'mpesa', 'm-pesa', 'mobile money', 'data revenue', 'subscriber growth',
                'network expansion', '5g', 'fiber optic', 'fuliza', 'bonga points',

                # Energy & Utilities
                'geothermal', 'renewable energy', 'grid expansion', 'power demand',
                'tariff approval', 'connection growth', 'energy security',

                # Regulatory & Policy
                'cbk approval', 'cma approval', 'license approval', 'regulatory clarity',
                'policy support', 'tax incentive', 'investment grade', 'credit rating',

                # Economic Indicators
                'gdp growth', 'inflation decline', 'shilling strength', 'forex reserves',
                'export growth', 'fdi inflow', 'infrastructure investment', 'ease of doing business',

                # Sectors
                'tea prices', 'coffee prices', 'tourism recovery', 'manufacturing growth',
                'construction boom', 'real estate upturn', 'agriculture boom'
            ],

            'negative': [
                # Market Performance
                'loss', 'decline', 'drop', 'crash', 'selloff', 'bearish', 'plunge',
                'downturn', 'correction', 'meltdown', 'panic', 'volatility', 'uncertainty',

                # Banking & Finance
                'loan defaults', 'npl increase', 'provision increase', 'margin compression',
                'credit crunch', 'liquidity crisis', 'capital shortfall', 'stress test failure',

                # Regulatory Issues
                'investigation', 'fine', 'penalty', 'suspension', 'delisting threat',
                'regulatory action', 'compliance failure', 'license revocation',

                # Economic Challenges
                'recession', 'inflation spike', 'shilling weakness', 'currency crisis',
                'debt crisis', 'fiscal deficit', 'current account deficit', 'drought impact',

                # Political & Social
                'political uncertainty', 'election concerns', 'strike', 'unrest',
                'security concerns', 'corruption scandal', 'governance issues',

                # Sector-Specific
                'power outages', 'fuel shortage', 'supply chain disruption',
                'commodity price decline', 'tourism decline', 'drought', 'flooding',

                # Company-Specific
                'profit warning', 'guidance cut', 'layoffs', 'restructuring',
                'debt default', 'bankruptcy risk', 'management changes', 'audit issues'
            ]
        }

        # Source reliability weights for Kenyan sources
        self.kenyan_source_weights = {
            'business_daily': 1.3,
            'kenyan_wallstreet': 1.2,
            'nse_kenya': 1.4,
            'cma_kenya': 1.4,
            'cbk_kenya': 1.3,
            'standard_digital': 1.1,
            'nation_media': 1.1,
            'citizen_digital': 1.0,
            'capital_fm': 1.0,
            'reuters_kenya': 1.2,
            'bloomberg_africa': 1.2,
            'african_business': 1.0,
            'social_media': 0.7,
            'reddit': 0.6,
            'twitter': 0.6
        }

        # Sector-specific sentiment modifiers
        self.sector_modifiers = {
            'Banking': 1.2,      # Banking news has high market impact
            'Telecommunications': 1.3,  # Safaricom dominates NSE
            'Manufacturing': 1.1,
            'Insurance': 1.0,
            'Utilities': 1.1,
            'Construction': 1.0,
            'Agriculture': 1.0,
            'Media': 0.9,
            'Investment Services': 1.0
        }

    def detect_news_type(self, content: str) -> str:
        """Detect the type of news for better context"""
        content_lower = content.lower()

        if any(word in content_lower for word in ['earnings', 'profit', 'revenue', 'financial results']):
            return 'earnings'
        elif any(word in content_lower for word in ['regulation', 'cbk', 'cma', 'policy', 'license']):
            return 'regulatory'
        elif any(word in content_lower for word in ['gdp', 'inflation', 'economy', 'government', 'budget']):
            return 'macroeconomic'
        elif any(word in content_lower for word in ['sector', 'industry', 'market segment']):
            return 'sector'
        else:
            return 'company'

    def calculate_kenyan_context_score(self, content: str, sector: str = None) -> float:
        """Calculate sentiment score based on Kenyan market keywords"""
        content_lower = content.lower()

        positive_matches = sum(1 for keyword in self.kenyan_financial_keywords['positive']
                             if keyword in content_lower)
        negative_matches = sum(1 for keyword in self.kenyan_financial_keywords['negative']
                             if keyword in content_lower)

        if positive_matches + negative_matches == 0:
            return 0.0

        base_score = (positive_matches - negative_matches) / (positive_matches + negative_matches)

        # Apply sector modifier if available
        if sector and sector in self.sector_modifiers:
            base_score *= self.sector_modifiers[sector]

        return max(-1.0, min(1.0, base_score))

    def analyze_sentiment(self, content: str, source: str = 'general',
                         sector: str = None, company_name: str = None) -> Tuple[float, float, str]:
        """
        Comprehensive sentiment analysis optimized for Kenyan market
        Returns: (sentiment_score, confidence, news_type)
        """
        if not content or len(content.strip()) < 5:
            return 0.5, 0.0, 'unknown'

        # Detect news type
        news_type = self.detect_news_type(content)

        # Multiple sentiment analysis approaches
        # 1. VADER sentiment
        vader_scores = self.sia.polarity_scores(content)
        vader_compound = vader_scores['compound']

        # 2. TextBlob sentiment
        try:
            blob = TextBlob(content)
            textblob_polarity = blob.sentiment.polarity
        except:
            textblob_polarity = 0.0

        # 3. Kenyan market context score
        context_score = self.calculate_kenyan_context_score(content, sector)

        # 4. Company-specific adjustments (if company name mentioned positively/negatively)
        company_modifier = 1.0
        if company_name and company_name.lower() in content.lower():
            # Look for direct positive/negative mentions of the company
            company_context = content.lower()
            if any(word in company_context for word in ['announces', 'launches', 'wins', 'grows']):
                company_modifier = 1.1
            elif any(word in company_context for word in ['loses', 'drops', 'fails', 'cuts']):
                company_modifier = 0.9

        # Combine scores with enhanced weights for Kenyan context
        weights = {
            'vader': 0.3,
            'textblob': 0.25,
            'context': 0.35,  # Higher weight for Kenyan context
            'company': 0.1
        }

        combined_score = (
            vader_compound * weights['vader'] +
            textblob_polarity * weights['textblob'] +
            context_score * weights['context']
        ) * company_modifier

        # Apply source reliability weight
        source_weight = self.kenyan_source_weights.get(source.lower(), 1.0)
        weighted_score = combined_score * source_weight

        # Convert to 0-1 scale (0-0.5 positive, 0.5-1.0 negative)
        if weighted_score >= 0:
            final_score = 0.5 - (weighted_score * 0.5)
        else:
            final_score = 0.5 + (abs(weighted_score) * 0.5)

        final_score = max(0.0, min(1.0, final_score))

        # Enhanced confidence calculation
        confidence_factors = [
            abs(weighted_score),  # Score strength
            min(len(content) / 200, 1.0),  # Content length
            source_weight / 1.4,  # Source reliability
            len([k for k in self.kenyan_financial_keywords['positive'] +
                self.kenyan_financial_keywords['negative'] if k in content.lower()]) / 10
        ]

        confidence = min(np.mean(confidence_factors), 1.0)

        return final_score, confidence, news_type

class AdvancedKenyanWebScraper:
    """Advanced web scraper specifically for Kenyan financial sources using Selenium"""

    def __init__(self, sentiment_analyzer: KenyanSentimentAnalyzer, db_manager: EnhancedDatabaseManager):
        self.sentiment_analyzer = sentiment_analyzer
        self.db_manager = db_manager
        self.session = requests.Session()

        # Enhanced headers to avoid detection
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.5',
            'Accept-Encoding': 'gzip, deflate, br',
            'Connection': 'keep-alive',
            'Upgrade-Insecure-Requests': '1',
        })

        self.driver = None
        self.setup_selenium()

    def setup_selenium(self):
        """Setup enhanced Selenium WebDriver for Kenyan sites"""
        try:
            chrome_options = Options()

            # Stealth options
            chrome_options.add_argument('--headless=new')  # Use new headless mode
            chrome_options.add_argument('--no-sandbox')
            chrome_options.add_argument('--disable-dev-shm-usage')
            chrome_options.add_argument('--disable-gpu')
            chrome_options.add_argument('--window-size=1920,1080')
            chrome_options.add_argument('--disable-blink-features=AutomationControlled')
            chrome_options.add_argument('--disable-extensions')
            chrome_options.add_argument('--disable-plugins')
            chrome_options.add_argument('--disable-images')  # Faster loading
            chrome_options.add_argument('--disable-javascript')  # For some static content

            # Additional stealth options
            chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
            chrome_options.add_experimental_option('useAutomationExtension', False)

            # Set user agent
            chrome_options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')

            self.driver = webdriver.Chrome(options=chrome_options)
            self.driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

            # Set implicit wait
            self.driver.implicitly_wait(10)

            logging.info("Enhanced Selenium WebDriver initialized successfully")

        except Exception as e:
            logging.warning(f"Selenium setup failed: {e}. Falling back to requests only.")
            self.driver = None

    async def scrape_all_kenyan_sources(self, symbols: List[str]) -> List[SentimentRecord]:
        """Scrape all Kenyan financial sources for given symbols"""
        all_records = []

        # Define Kenyan source scrapers
        scrapers = [
            self.scrape_business_daily,
            self.scrape_standard_digital,
            self.scrape_nation_media,
            self.scrape_kenyan_wallstreet,
            self.scrape_citizen_digital,
            self.scrape_capital_fm,
            self.scrape_nse_announcements,
        ]

        # Use ThreadPoolExecutor for concurrent scraping
        with ThreadPoolExecutor(max_workers=3) as executor:
            future_to_scraper = {}

            for symbol in symbols:
                for scraper in scrapers:
                    future = executor.submit(scraper, symbol)
                    future_to_scraper[future] = (scraper.__name__, symbol)

            for future in as_completed(future_to_scraper):
                scraper_name, symbol = future_to_scraper[future]
                try:
                    records = future.result()
                    all_records.extend(records)
                    logging.info(f"{scraper_name} completed for {symbol}: {len(records)} records")
                except Exception as e:
                    logging.error(f"Error in {scraper_name} for {symbol}: {e}")

        return all_records

    def scrape_business_daily(self, symbol: str) -> List[SentimentRecord]:
        """Scrape Business Daily Africa"""
        records = []
        company_info = self.db_manager.get_company_info(symbol)

        try:
            # Business Daily search URL
            search_terms = [symbol, company_info['company_name']]

            for term in search_terms:
                if not term or term == symbol:
                    continue

                url = f"https://www.businessdailyafrica.com/bd/search?q={term}"

                if self.driver:
                    self.driver.get(url)
                    time.sleep(3)

                    # Look for article elements
                    articles = self.driver.find_elements(By.CSS_SELECTOR, "article, .article, .story, .news-item")

                    for article in articles[:5]:  # Limit to 5 most recent
                        try:
                            title_elem = article.find_element(By.CSS_SELECTOR, "h1, h2, h3, .title, .headline")
                            title = title_elem.text.strip()

                            # Try to get article link
                            link_elem = article.find_element(By.CSS_SELECTOR, "a")
                            link = link_elem.get_attribute('href') if link_elem else ""

                            # Try to get summary/excerpt
                            try:
                                summary_elem = article.find_element(By.CSS_SELECTOR, ".summary, .excerpt, p")
                                summary = summary_elem.text.strip()
                            except:
                                summary = ""

                            content = f"{title}. {summary}"

                            if len(content.strip()) > 20:
                                sentiment_score, confidence, news_type = self.sentiment_analyzer.analyze_sentiment(
                                    content, 'business_daily', company_info['sector'], company_info['company_name']
                                )

                                records.append(SentimentRecord(
                                    timestamp=datetime.now(),
                                    symbol=symbol,
                                    company_name=company_info['company_name'],
                                    source='business_daily',
                                    content=content,
                                    sentiment_score=sentiment_score,
                                    confidence=confidence,
                                    category='news',
                                    url=link,
                                    sector=company_info['sector'],
                                    news_type=news_type
                                ))
                        except Exception as e:
                            continue

                time.sleep(2)  # Rate limiting

        except Exception as e:
            logging.error(f"Error scraping Business Daily for {symbol}: {e}")

        return records

    def scrape_standard_digital(self, symbol: str) -> List[SentimentRecord]:
        """Scrape Standard Digital business section"""
        records = []
        company_info = self.db_manager.get_company_info(symbol)

        try:
            base_url = "https://www.standardmedia.co.ke/business"

            if self.driver:
                self.driver.get(base_url)
                time.sleep(3)

                # Look for business articles
                articles = self.driver.find_elements(By.CSS_SELECTOR, ".article, .story, .news-item, .post")

                for article in articles[:8]:
                    try:
                        title_elem = article.find_element(By.CSS_SELECTOR, "h1, h2, h3, .title, .headline, a")
                        title = title_elem.text.strip()

                        # Check if article is relevant to the symbol/company
                        if (symbol.lower() in title.lower() or
                            company_info['company_name'].lower() in title.lower() or
                            any(keyword in title.lower() for keyword in ['bank', 'telecom', 'insurance', 'market', 'stock'])):

                            link = title_elem.get_attribute('href') if title_elem.tag_name == 'a' else ""

                            # Get article summary if available
                            try:
                                summary_elem = article.find_element(By.CSS_SELECTOR, ".summary, .excerpt, .description, p")
                                summary = summary_elem.text.strip()
                            except:
                                summary = ""

                            content = f"{title}. {summary}"

                            if len(content.strip()) > 20:
                                sentiment_score, confidence, news_type = self.sentiment_analyzer.analyze_sentiment(
                                    content, 'standard_digital', company_info['sector'], company_info['company_name']
                                )

                                records.append(SentimentRecord(
                                    timestamp=datetime.now(),
                                    symbol=symbol,
                                    company_name=company_info['company_name'],
                                    source='standard_digital',
                                    content=content,
                                    sentiment_score=sentiment_score,
                                    confidence=confidence,
                                    category='news',
                                    url=link,
                                    sector=company_info['sector'],
                                    news_type=news_type
                                ))
                    except Exception as e:
                        continue

        except Exception as e:
            logging.error(f"Error scraping Standard Digital for {symbol}: {e}")

        return records

    def scrape_nation_media(self, symbol: str) -> List[SentimentRecord]:
        """Scrape Nation Media business section"""
        records = []
        company_info = self.db_manager.get_company_info(symbol)

        try:
            base_url = "https://nation.africa/kenya/business"

            if self.driver:
                self.driver.get(base_url)
                time.sleep(3)

                articles = self.driver.find_elements(By.CSS_SELECTOR, ".teaser, .article-teaser, .story-item")

                for article in articles[:8]:
                    try:
                        title_elem = article.find_element(By.CSS_SELECTOR, "h1, h2, h3, .title, .headline")
                        title = title_elem.text.strip()

                        if (symbol.lower() in title.lower() or
                            company_info['company_name'].lower() in title.lower() or
                            any(keyword in title.lower() for keyword in ['market', 'shares', 'profit', 'earnings', 'nse'])):

                            try:
                                link_elem = article.find_element(By.CSS_SELECTOR, "a")
                                link = link_elem.get_attribute('href')
                            except:
                                link = ""

                            try:
                                summary_elem = article.find_element(By.CSS_SELECTOR, ".summary, .excerpt, p")
                                summary = summary_elem.text.strip()
                            except:
                                summary = ""

                            content = f"{title}. {summary}"

                            if len(content.strip()) > 20:
                                sentiment_score, confidence, news_type = self.sentiment_analyzer.analyze_sentiment(
                                    content, 'nation_media', company_info['sector'], company_info['company_name']
                                )

                                records.append(SentimentRecord(
                                    timestamp=datetime.now(),
                                    symbol=symbol,
                                    company_name=company_info['company_name'],
                                    source='nation_media',
                                    content=content,
                                    sentiment_score=sentiment_score,
                                    confidence=confidence,
                                    category='news',
                                    url=link,
                                    sector=company_info['sector'],
                                    news_type=news_type
                                ))
                    except Exception as e:
                        continue

        except Exception as e:
            logging.error(f"Error scraping Nation Media for {symbol}: {e}")

        return records

    def scrape_kenyan_wallstreet(self, symbol: str) -> List[SentimentRecord]:
        """Scrape Kenyan Wall Street - specialized financial site"""
        records = []
        company_info = self.db_manager.get_company_info(symbol)

        try:
            base_url = "https://kenyanwallstreet.com"
            search_url = f"{base_url}/search?q={symbol}"

            if self.driver:
                self.driver.get(search_url)
                time.sleep(4)

                # Look for financial news articles
                articles = self.driver.find_elements(By.CSS_SELECTOR, ".post, .article, .entry, .news-item")

                for article in articles[:10]:  # More articles from specialized source
                    try:
                        title_elem = article.find_element(By.CSS_SELECTOR, "h1, h2, h3, .title, .post-title")
                        title = title_elem.text.strip()

                        try:
                            link_elem = article.find_element(By.CSS_SELECTOR, "a")
                            link = link_elem.get_attribute('href')
                        except:
                            link = ""

                        try:
                            summary_elem = article.find_element(By.CSS_SELECTOR, ".excerpt, .summary, .content, p")
                            summary = summary_elem.text.strip()
                        except:
                            summary = ""

                        content = f"{title}. {summary}"

                        if len(content.strip()) > 20:
                            sentiment_score, confidence, news_type = self.sentiment_analyzer.analyze_sentiment(
                                content, 'kenyan_wallstreet', company_info['sector'], company_info['company_name']
                            )

                            records.append(SentimentRecord(
                                timestamp=datetime.now(),
                                symbol=symbol,
                                company_name=company_info['company_name'],
                                source='kenyan_wallstreet',
                                content=content,
                                sentiment_score=sentiment_score,
                                confidence=confidence,
                                category='financial_news',
                                url=link,
                                sector=company_info['sector'],
                                news_type=news_type,
                                market_cap_impact=1.2  # Higher impact from specialized source
                            ))
                    except Exception as e:
                        continue

        except Exception as e:
            logging.error(f"Error scraping Kenyan Wall Street for {symbol}: {e}")

        return records

    def scrape_citizen_digital(self, symbol: str) -> List[SentimentRecord]:
        """Scrape Citizen Digital business section"""
        records = []
        company_info = self.db_manager.get_company_info(symbol)

        try:
            base_url = "https://www.citizen.digital/business"

            if self.driver:
                self.driver.get(base_url)
                time.sleep(3)

                articles = self.driver.find_elements(By.CSS_SELECTOR, ".post, .article-item, .news-card")

                for article in articles[:6]:
                    try:
                        title_elem = article.find_element(By.CSS_SELECTOR, "h1, h2, h3, .title")
                        title = title_elem.text.strip()

                        if (symbol.lower() in title.lower() or
                            company_info['company_name'].lower() in title.lower() or
                            any(keyword in title.lower() for keyword in ['business', 'market', 'company', 'profit'])):

                            try:
                                link_elem = article.find_element(By.CSS_SELECTOR, "a")
                                link = link_elem.get_attribute('href')
                            except:
                                link = ""

                            try:
                                summary_elem = article.find_element(By.CSS_SELECTOR, ".excerpt, p")
                                summary = summary_elem.text.strip()
                            except:
                                summary = ""

                            content = f"{title}. {summary}"

                            if len(content.strip()) > 20:
                                sentiment_score, confidence, news_type = self.sentiment_analyzer.analyze_sentiment(
                                    content, 'citizen_digital', company_info['sector'], company_info['company_name']
                                )

                                records.append(SentimentRecord(
                                    timestamp=datetime.now(),
                                    symbol=symbol,
                                    company_name=company_info['company_name'],
                                    source='citizen_digital',
                                    content=content,
                                    sentiment_score=sentiment_score,
                                    confidence=confidence,
                                    category='news',
                                    url=link,
                                    sector=company_info['sector'],
                                    news_type=news_type
                                ))
                    except Exception as e:
                        continue

        except Exception as e:
            logging.error(f"Error scraping Citizen Digital for {symbol}: {e}")

        return records

    def scrape_capital_fm(self, symbol: str) -> List[SentimentRecord]:
        """Scrape Capital FM business section"""
        records = []
        company_info = self.db_manager.get_company_info(symbol)

        try:
            base_url = "https://www.capitalfm.co.ke/business"

            if self.driver:
                self.driver.get(base_url)
                time.sleep(3)

                articles = self.driver.find_elements(By.CSS_SELECTOR, ".post, .story, .article")

                for article in articles[:6]:
                    try:
                        title_elem = article.find_element(By.CSS_SELECTOR, "h1, h2, h3, .title, .post-title")
                        title = title_elem.text.strip()

                        if (symbol.lower() in title.lower() or
                            company_info['company_name'].lower() in title.lower()):

                            try:
                                link_elem = article.find_element(By.CSS_SELECTOR, "a")
                                link = link_elem.get_attribute('href')
                            except:
                                link = ""

                            content = title  # Capital FM usually has shorter excerpts

                            if len(content.strip()) > 15:
                                sentiment_score, confidence, news_type = self.sentiment_analyzer.analyze_sentiment(
                                    content, 'capital_fm', company_info['sector'], company_info['company_name']
                                )

                                records.append(SentimentRecord(
                                    timestamp=datetime.now(),
                                    symbol=symbol,
                                    company_name=company_info['company_name'],
                                    source='capital_fm',
                                    content=content,
                                    sentiment_score=sentiment_score,
                                    confidence=confidence,
                                    category='news',
                                    url=link,
                                    sector=company_info['sector'],
                                    news_type=news_type
                                ))
                    except Exception as e:
                        continue

        except Exception as e:
            logging.error(f"Error scraping Capital FM for {symbol}: {e}")

        return records

    def scrape_nse_announcements(self, symbol: str) -> List[SentimentRecord]:
        """Scrape NSE official announcements - highest priority source"""
        records = []
        company_info = self.db_manager.get_company_info(symbol)

        try:
            base_url = "https://www.nse.co.ke/listed-companies/company-announcements"

            if self.driver:
                self.driver.get(base_url)
                time.sleep(4)

                # Look for announcement tables or lists
                announcements = self.driver.find_elements(By.CSS_SELECTOR, ".announcement, .company-news, tr, .news-item")

                for announcement in announcements[:5]:  # Official announcements are more valuable
                    try:
                        # Try different selectors for title/content
                        text_elem = announcement.find_element(By.CSS_SELECTOR, "td, .title, .content, .text")
                        text = text_elem.text.strip()

                        if symbol in text or company_info['company_name'].lower() in text.lower():
                            try:
                                link_elem = announcement.find_element(By.CSS_SELECTOR, "a")
                                link = link_elem.get_attribute('href')
                            except:
                                link = base_url

                            if len(text.strip()) > 20:
                                sentiment_score, confidence, news_type = self.sentiment_analyzer.analyze_sentiment(
                                    text, 'nse_kenya', company_info['sector'], company_info['company_name']
                                )

                                records.append(SentimentRecord(
                                    timestamp=datetime.now(),
                                    symbol=symbol,
                                    company_name=company_info['company_name'],
                                    source='nse_kenya',
                                    content=text,
                                    sentiment_score=sentiment_score,
                                    confidence=confidence,
                                    category='official_announcement',
                                    url=link,
                                    sector=company_info['sector'],
                                    news_type='regulatory',
                                    market_cap_impact=1.5  # Highest impact from official source
                                ))
                    except Exception as e:
                        continue

        except Exception as e:
            logging.error(f"Error scraping NSE for {symbol}: {e}")

        return records

    def scrape_social_media_mentions(self, symbol: str) -> List[SentimentRecord]:
        """Scrape social media mentions from various platforms"""
        records = []
        company_info = self.db_manager.get_company_info(symbol)

        # Twitter/X scraping (public posts only)
        try:
            search_terms = [symbol, company_info['company_name']]

            for term in search_terms:
                if not term or len(term) < 2:
                    continue

                # Use Nitter or similar public interfaces
                nitter_url = f"https://nitter.net/search?q={term}+kenya+stock+market"

                if self.driver:
                    self.driver.get(nitter_url)
                    time.sleep(3)

                    tweets = self.driver.find_elements(By.CSS_SELECTOR, ".tweet-content, .tweet-text")

                    for tweet in tweets[:5]:
                        try:
                            tweet_text = tweet.text.strip()

                            if len(tweet_text) > 20 and (symbol.lower() in tweet_text.lower() or
                                                        company_info['company_name'].lower() in tweet_text.lower()):

                                sentiment_score, confidence, news_type = self.sentiment_analyzer.analyze_sentiment(
                                    tweet_text, 'twitter', company_info['sector'], company_info['company_name']
                                )

                                records.append(SentimentRecord(
                                    timestamp=datetime.now(),
                                    symbol=symbol,
                                    company_name=company_info['company_name'],
                                    source='twitter',
                                    content=tweet_text,
                                    sentiment_score=sentiment_score,
                                    confidence=confidence,
                                    category='social_media',
                                    sector=company_info['sector'],
                                    news_type=news_type,
                                    engagement_score=0.5  # Default engagement
                                ))
                        except Exception as e:
                            continue

        except Exception as e:
            logging.error(f"Error scraping social media for {symbol}: {e}")

        return records

    def close_selenium(self):
        """Close Selenium WebDriver with error handling"""
        if self.driver:
            try:
                self.driver.quit()
                logging.info("Selenium WebDriver closed successfully")
            except Exception as e:
                logging.error(f"Error closing Selenium WebDriver: {e}")
            finally:
                self.driver = None

    def __del__(self):
        """Cleanup when object is destroyed"""
        self.close_selenium()

class KenyanSentimentAggregator:
    """Enhanced aggregator for Kenyan market sentiment with predictive features"""

    def __init__(self, db_manager: EnhancedDatabaseManager):
        self.db_manager = db_manager

    def calculate_comprehensive_metrics(self, symbol: str, period_hours: int = 3) -> Dict:
        """Calculate comprehensive metrics for Kenyan market analysis"""
        df = self.get_enhanced_recent_data(symbol, period_hours)

        if df.empty:
            return self.get_default_metrics()

        company_info = self.db_manager.get_company_info(symbol)

        # Basic sentiment metrics
        basic_metrics = self.calculate_basic_sentiment_metrics(df)

        # Advanced Kenyan market metrics
        advanced_metrics = self.calculate_advanced_kenyan_metrics(df, company_info)

        # Predictive indicators
        predictive_metrics = self.calculate_predictive_indicators(df, symbol)

        # Combine all metrics
        return {**basic_metrics, **advanced_metrics, **predictive_metrics}

    def get_enhanced_recent_data(self, symbol: str, period_hours: int) -> pd.DataFrame:
        """Get enhanced recent data with source reliability weighting"""
        conn = sqlite3.connect(self.db_manager.db_path)

        query = '''
            SELECT sd.*, ds.reliability_score, ds.kenyan_focus
            FROM sentiment_data sd
            LEFT JOIN data_sources ds ON sd.source = ds.source_name
            WHERE sd.symbol = ? AND sd.timestamp > datetime('now', '-{} hours')
            ORDER BY sd.timestamp DESC
        '''.format(period_hours)

        df = pd.read_sql_query(query, conn, params=(symbol,))
        conn.close()
        return df

    def calculate_basic_sentiment_metrics(self, df: pd.DataFrame) -> Dict:
        """Calculate basic sentiment metrics"""
        total_mentions = len(df)

        if total_mentions == 0:
            return self.get_default_metrics()

        positive_count = len(df[df['sentiment_score'] < 0.5])
        negative_count = len(df[df['sentiment_score'] > 0.5])
        neutral_count = len(df[df['sentiment_score'] == 0.5])

        average_sentiment = df['sentiment_score'].mean()
        confidence_weighted_score = np.average(df['sentiment_score'], weights=df['confidence'])
        volatility_score = df['sentiment_score'].std()

        return {
            'total_mentions': total_mentions,
            'positive_count': positive_count,
            'negative_count': negative_count,
            'neutral_count': neutral_count,
            'average_sentiment': average_sentiment,
            'confidence_weighted_score': confidence_weighted_score,
            'volatility_score': volatility_score if not np.isnan(volatility_score) else 0.0
        }

    def calculate_advanced_kenyan_metrics(self, df: pd.DataFrame, company_info: Dict) -> Dict:
        """Calculate advanced metrics specific to Kenyan market"""
        # Source reliability weighted score
        if 'reliability_score' in df.columns:
            reliability_weights = df['reliability_score'].fillna(1.0)
            reliability_weighted_score = np.average(df['sentiment_score'], weights=reliability_weights)
        else:
            reliability_weighted_score = df['sentiment_score'].mean()

        # Kenyan source focus score (higher weight for local sources)
        kenyan_focused = df[df['kenyan_focus'] == True] if 'kenyan_focus' in df.columns else df
        kenyan_sentiment = kenyan_focused['sentiment_score'].mean() if not kenyan_focused.empty else 0.5

        # Sector-specific analysis
        sector = company_info.get('sector', 'Unknown')
        sector_sentiment = self.calculate_sector_context(df, sector)

        # News type distribution and impact
        news_type_impact = self.calculate_news_type_impact(df)

        # Regulatory and market context
        regulatory_impact = self.calculate_regulatory_impact(df)
        market_momentum = self.calculate_market_momentum(df)

        return {
            'sector_sentiment': sector_sentiment,
            'kenyan_focused_sentiment': kenyan_sentiment,
            'reliability_weighted_score': reliability_weighted_score,
            'market_momentum': market_momentum,
            'regulatory_impact': regulatory_impact,
            'news_type_distribution': news_type_impact,
            'economic_context': self.get_economic_context()
        }

    def calculate_sector_context(self, df: pd.DataFrame, sector: str) -> float:
        """Calculate sector-specific sentiment context"""
        if sector == 'Banking':
            banking_keywords = ['loan', 'deposit', 'npl', 'provision', 'interest', 'cbk']
            sector_df = df[df['content'].str.contains('|'.join(banking_keywords), case=False, na=False)]
        elif sector == 'Telecommunications':
            telecom_keywords = ['mpesa', 'mobile', 'data', 'subscriber', 'network', '5g']
            sector_df = df[df['content'].str.contains('|'.join(telecom_keywords), case=False, na=False)]
        elif sector == 'Manufacturing':
            manuf_keywords = ['production', 'factory', 'export', 'supply', 'manufacturing']
            sector_df = df[df['content'].str.contains('|'.join(manuf_keywords), case=False, na=False)]
        else:
            sector_df = df

        return sector_df['sentiment_score'].mean() if not sector_df.empty else 0.5

    def calculate_news_type_impact(self, df: pd.DataFrame) -> Dict:
        """Calculate impact based on news type distribution"""
        news_types = df['news_type'].value_counts().to_dict() if 'news_type' in df.columns else {}

        # Weight different news types by market impact
        type_weights = {
            'earnings': 1.5,
            'regulatory': 1.4,
            'macroeconomic': 1.2,
            'sector': 1.1,
            'company': 1.0
        }

        weighted_impact = 0
        total_weight = 0

        for news_type, count in news_types.items():
            weight = type_weights.get(news_type, 1.0)
            type_sentiment = df[df['news_type'] == news_type]['sentiment_score'].mean()
            weighted_impact += type_sentiment * weight * count
            total_weight += weight * count

        return {
            'distribution': news_types,
            'weighted_sentiment_impact': weighted_impact / total_weight if total_weight > 0 else 0.5
        }

    def calculate_regulatory_impact(self, df: pd.DataFrame) -> float:
        """Calculate regulatory sentiment impact"""
        regulatory_keywords = ['cbk', 'cma', 'regulation', 'compliance', 'license', 'approval']
        regulatory_df = df[df['content'].str.contains('|'.join(regulatory_keywords), case=False, na=False)]

        if regulatory_df.empty:
            return 0.5

        # Regulatory news has higher impact, so amplify the deviation from neutral
        reg_sentiment = regulatory_df['sentiment_score'].mean()
        return reg_sentiment

    def calculate_market_momentum(self, df: pd.DataFrame) -> float:
        """Calculate market momentum based on recent sentiment trend"""
        if len(df) < 3:
            return 0.5

        df_sorted = df.sort_values('timestamp')
        recent_third = df_sorted.tail(len(df) // 3)
        earlier_third = df_sorted.head(len(df) // 3)

        recent_avg = recent_third['sentiment_score'].mean()
        earlier_avg = earlier_third['sentiment_score'].mean()

        # Momentum: positive if sentiment improving, negative if deteriorating
        momentum = (earlier_avg - recent_avg)  # Inverted because lower score = more positive

        # Normalize to 0-1 scale
        return max(0, min(1, 0.5 + momentum))

    def calculate_predictive_indicators(self, df: pd.DataFrame, symbol: str) -> Dict:
        """Calculate indicators for predictive modeling"""
        if df.empty:
            return {'prediction_confidence': 0.0, 'trend_strength': 0.0, 'volatility_risk': 0.5}

        # Trend strength based on sentiment consistency
        sentiment_values = df['sentiment_score'].values
        if len(sentiment_values) > 1:
            trend_strength = 1 - (np.std(sentiment_values) / 0.5)  # Normalize by max possible std
            trend_strength = max(0, min(1, trend_strength))
        else:
            trend_strength = 0.0

        # Prediction confidence based on multiple factors
        factors = [
            len(df) / 20,  # Sample size factor
            df['confidence'].mean(),  # Average confidence
            trend_strength,  # Trend consistency
            len(df['source'].unique()) / 5  # Source diversity
        ]

        prediction_confidence = min(1.0, np.mean(factors))

        # Volatility risk assessment
        recent_volatility = df['sentiment_score'].rolling(window=3, min_periods=1).std().mean()
        volatility_risk = min(1.0, recent_volatility * 2) if not np.isnan(recent_volatility) else 0.5

        return {
            'prediction_confidence': prediction_confidence,
            'trend_strength': trend_strength,
            'volatility_risk': volatility_risk,
            'trending_topics': self.extract_kenyan_trending_topics(df['content'].tolist())
        }

    def get_economic_context(self) -> float:
        """Get current economic context for Kenya (placeholder for real economic data)"""
        # In production, this would fetch real economic indicators
        # For now, return neutral context
        return 0.5

    def extract_kenyan_trending_topics(self, texts: List[str], max_features: int = 10) -> List[str]:
        """Extract trending topics with Kenyan market focus"""
        if not texts:
            return []

        try:
            combined_text = ' '.join(texts)

            # Kenyan-specific stopwords
            kenyan_stopwords = set(['kenya', 'kenyan', 'nairobi', 'nse', 'shilling', 'ksh'])
            english_stopwords = set(stopwords.words('english'))
            all_stopwords = english_stopwords.union(kenyan_stopwords)

            vectorizer = TfidfVectorizer(
                max_features=max_features,
                stop_words=list(all_stopwords),
                ngram_range=(1, 2),
                min_df=1,
                max_df=0.8
            )

            tfidf_matrix = vectorizer.fit_transform([combined_text])
            feature_names = vectorizer.get_feature_names_out()
            tfidf_scores = tfidf_matrix.toarray()[0]

            top_indices = tfidf_scores.argsort()[-max_features:][::-1]
            trending_topics = [feature_names[i] for i in top_indices if tfidf_scores[i] > 0]

            return trending_topics[:5]

        except Exception as e:
            logging.error(f"Error extracting trending topics: {e}")
            return []

    def get_default_metrics(self) -> Dict:
        """Return default metrics when no data available"""
        return {
            'total_mentions': 0,
            'positive_count': 0,
            'negative_count': 0,
            'neutral_count': 0,
            'average_sentiment': 0.5,
            'confidence_weighted_score': 0.5,
            'volatility_score': 0.0,
            'sector_sentiment': 0.5,
            'kenyan_focused_sentiment': 0.5,
            'reliability_weighted_score': 0.5,
            'market_momentum': 0.5,
            'regulatory_impact': 0.5,
            'news_type_distribution': {},
            'economic_context': 0.5,
            'prediction_confidence': 0.0,
            'trend_strength': 0.0,
            'volatility_risk': 0.5,
            'trending_topics': []
        }

    def save_aggregated_score(self, symbol: str, metrics: Dict):
        """Save comprehensive aggregated score to database"""
        try:
            conn = sqlite3.connect(self.db_manager.db_path)
            cursor = conn.cursor()

            company_info = self.db_manager.get_company_info(symbol)
            now = datetime.now()
            period_start = now - timedelta(hours=3)

            cursor.execute('''
                INSERT INTO aggregated_scores
                (timestamp, symbol, company_name, period_start, period_end, average_sentiment,
                 positive_count, negative_count, neutral_count, total_mentions,
                 confidence_weighted_score, volatility_score, trending_topics, sector_sentiment,
                 market_momentum, regulatory_impact, economic_context, prediction_confidence)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ''', (
                now, symbol, company_info['company_name'], period_start, now,
                metrics['average_sentiment'], metrics['positive_count'],
                metrics['negative_count'], metrics['neutral_count'], metrics['total_mentions'],
                metrics['confidence_weighted_score'], metrics['volatility_score'],
                json.dumps(metrics['trending_topics']), metrics['sector_sentiment'],
                metrics['market_momentum'], metrics['regulatory_impact'],
                metrics['economic_context'], metrics['prediction_confidence']
            ))

            conn.commit()
            conn.close()

            logging.info(f"Comprehensive metrics saved for {symbol}: Score {metrics['confidence_weighted_score']:.3f}, Confidence {metrics['prediction_confidence']:.3f}")

        except Exception as e:
            logging.error(f"Error saving aggregated score: {e}")

class KenyanStockSentimentSystem:
    """Enhanced main system for Kenyan market sentiment analysis"""

    def __init__(self, symbols: List[str], db_path: str = "kenyan_market_sentiment.db",
                 collection_interval: int = 30, aggregation_interval: int = 180):
        self.symbols = symbols
        self.db_manager = EnhancedDatabaseManager(db_path)
        self.sentiment_analyzer = KenyanSentimentAnalyzer()
        self.web_scraper = AdvancedKenyanWebScraper(self.sentiment_analyzer, self.db_manager)
        self.aggregator = KenyanSentimentAggregator(self.db_manager)

        self.collection_interval = collection_interval  # minutes
        self.aggregation_interval = aggregation_interval  # minutes
        self.is_running = False
        self.collection_stats = {'total_cycles': 0, 'total_records': 0, 'failed_cycles': 0}

    async def collect_comprehensive_data(self, symbol: str) -> Dict:
        """Collect comprehensive data for a single symbol with detailed tracking"""
        logging.info(f"Starting comprehensive data collection for {symbol}...")

        collection_results = {
            'symbol': symbol,
            'timestamp': datetime.now(),
            'sources_scraped': [],
            'records_collected': 0,
            'errors': []
        }

        try:
            company_info = self.db_manager.get_company_info(symbol)

            # Collect from all Kenyan sources
            all_records = await self.web_scraper.scrape_all_kenyan_sources([symbol])

            # Collect social media mentions
            social_records = self.web_scraper.scrape_social_media_mentions(symbol)
            all_records.extend(social_records)

            # Save to database with detailed tracking
            saved_count = 0
            for record in all_records:
                if self.db_manager.insert_sentiment(record):
                    saved_count += 1
                    if record.source not in collection_results['sources_scraped']:
                        collection_results['sources_scraped'].append(record.source)

            collection_results['records_collected'] = saved_count

            # Log collection summary
            logging.info(f"Collection complete for {symbol}: {saved_count}/{len(all_records)} records saved from {len(collection_results['sources_scraped'])} sources")

            return collection_results

        except Exception as e:
            error_msg = f"Error in comprehensive data collection for {symbol}: {e}"
            logging.error(error_msg)
            collection_results['errors'].append(error_msg)
            return collection_results

    async def run_full_collection_cycle(self) -> Dict:
        """Run complete data collection cycle for all symbols with performance tracking"""
        cycle_start = datetime.now()
        logging.info("=" * 60)
        logging.info("STARTING KENYAN MARKET SENTIMENT COLLECTION CYCLE")
        logging.info("=" * 60)

        cycle_results = {
            'cycle_start': cycle_start,
            'symbols_processed': [],
            'total_records': 0,
            'sources_active': set(),
            'errors': [],
            'performance_metrics': {}
        }

        # Process each symbol
        for symbol in self.symbols:
            try:
                symbol_results = await self.collect_comprehensive_data(symbol)
                cycle_results['symbols_processed'].append(symbol_results)
                cycle_results['total_records'] += symbol_results['records_collected']
                cycle_results['sources_active'].update(symbol_results['sources_scraped'])
                cycle_results['errors'].extend(symbol_results['errors'])

                # Add delay between symbols to respect rate limits
                await asyncio.sleep(5)

            except Exception as e:
                error_msg = f"Failed to process {symbol}: {e}"
                logging.error(error_msg)
                cycle_results['errors'].append(error_msg)

        # Calculate cycle performance
        cycle_duration = (datetime.now() - cycle_start).total_seconds()
        cycle_results['performance_metrics'] = {
            'duration_seconds': cycle_duration,
            'records_per_minute': (cycle_results['total_records'] / cycle_duration) * 60 if cycle_duration > 0 else 0,
            'symbols_per_minute': (len(self.symbols) / cycle_duration) * 60 if cycle_duration > 0 else 0,
            'active_sources': len(cycle_results['sources_active']),
            'error_rate': len(cycle_results['errors']) / (len(self.symbols) + len(cycle_results['errors'])) if len(self.symbols) > 0 else 0
        }

        # Update collection stats
        self.collection_stats['total_cycles'] += 1
        self.collection_stats['total_records'] += cycle_results['total_records']
        if cycle_results['errors']:
            self.collection_stats['failed_cycles'] += 1

        # Log cycle summary
        logging.info("=" * 60)
        logging.info("COLLECTION CYCLE COMPLETE")
        logging.info(f"Duration: {cycle_duration:.1f}s | Records: {cycle_results['total_records']} | Sources: {len(cycle_results['sources_active'])}")
        logging.info(f"Performance: {cycle_results['performance_metrics']['records_per_minute']:.1f} records/min")
        if cycle_results['errors']:
            logging.warning(f"Errors encountered: {len(cycle_results['errors'])}")
        logging.info("=" * 60)

        return cycle_results

    def calculate_and_save_all_aggregations(self) -> Dict:
        """Calculate and save aggregated scores for all symbols with enhanced metrics"""
        logging.info("CALCULATING COMPREHENSIVE SENTIMENT AGGREGATIONS")

        aggregation_results = {
            'timestamp': datetime.now(),
            'symbols_processed': [],
            'aggregation_summary': {},
            'market_overview': {}
        }

        symbol_summaries = []

        for symbol in self.symbols:
            try:
                # Calculate comprehensive metrics
                metrics = self.aggregator.calculate_comprehensive_metrics(symbol)
                self.aggregator.save_aggregated_score(symbol, metrics)

                # Create symbol summary
                company_info = self.db_manager.get_company_info(symbol)
                symbol_summary = {
                    'symbol': symbol,
                    'company_name': company_info['company_name'],
                    'sector': company_info['sector'],
                    'sentiment_score': metrics['confidence_weighted_score'],
                    'prediction_confidence': metrics['prediction_confidence'],
                    'total_mentions': metrics['total_mentions'],
                    'trend_strength': metrics['trend_strength'],
                    'volatility_risk': metrics['volatility_risk'],
                    'sentiment_status': self.get_sentiment_status(metrics['confidence_weighted_score']),
                    'market_momentum': metrics['market_momentum']
                }

                symbol_summaries.append(symbol_summary)
                aggregation_results['symbols_processed'].append(symbol)

                # Log individual symbol status
                status = symbol_summary['sentiment_status']
                logging.info(f"{symbol} ({company_info['company_name']}): {status} | Score: {metrics['confidence_weighted_score']:.3f} | Mentions: {metrics['total_mentions']} | Confidence: {metrics['prediction_confidence']:.3f}")

            except Exception as e:
                logging.error(f"Error calculating aggregation for {symbol}: {e}")

        # Generate market overview
        aggregation_results['market_overview'] = self.generate_market_overview(symbol_summaries)
        aggregation_results['aggregation_summary'] = {
            'total_symbols': len(symbol_summaries),
            'positive_sentiment': len([s for s in symbol_summaries if s['sentiment_score'] < 0.5]),
            'negative_sentiment': len([s for s in symbol_summaries if s['sentiment_score'] > 0.5]),
            'neutral_sentiment': len([s for s in symbol_summaries if s['sentiment_score'] == 0.5]),
            'average_confidence': np.mean([s['prediction_confidence'] for s in symbol_summaries]) if symbol_summaries else 0,
            'total_mentions': sum([s['total_mentions'] for s in symbol_summaries]),
            'high_confidence_signals': len([s for s in symbol_summaries if s['prediction_confidence'] > 0.7])
        }

        return aggregation_results

    def get_sentiment_status(self, score: float) -> str:
        """Convert sentiment score to readable status"""
        if score < 0.3:
            return "VERY POSITIVE"
        elif score < 0.5:
            return "POSITIVE"
        elif score == 0.5:
            return "NEUTRAL"
        elif score < 0.7:
            return "NEGATIVE"
        else:
            return "VERY NEGATIVE"

    def generate_market_overview(self, symbol_summaries: List[Dict]) -> Dict:
        """Generate comprehensive market overview"""
        if not symbol_summaries:
            return {'status': 'NO_DATA'}

        # Sector analysis
        sector_sentiment = {}
        for summary in symbol_summaries:
            sector = summary['sector']
            if sector not in sector_sentiment:
                sector_sentiment[sector] = []
            sector_sentiment[sector].append(summary['sentiment_score'])

        sector_analysis = {}
        for sector, scores in sector_sentiment.items():
            sector_analysis[sector] = {
                'average_sentiment': np.mean(scores),
                'companies_count': len(scores),
                'status': self.get_sentiment_status(np.mean(scores))
            }

        # Market leaders and laggards
        sorted_by_sentiment = sorted(symbol_summaries, key=lambda x: x['sentiment_score'])

        market_overview = {
            'overall_market_sentiment': np.mean([s['sentiment_score'] for s in symbol_summaries]),
            'market_status': self.get_sentiment_status(np.mean([s['sentiment_score'] for s in symbol_summaries])),
            'sector_analysis': sector_analysis,
            'top_performers': sorted_by_sentiment[:3],  # Most positive
            'bottom_performers': sorted_by_sentiment[-3:],  # Most negative
            'highest_confidence': sorted(symbol_summaries, key=lambda x: x['prediction_confidence'], reverse=True)[:3],
            'most_volatile': sorted(symbol_summaries, key=lambda x: x['volatility_risk'], reverse=True)[:3],
            'trending_symbols': [s for s in symbol_summaries if s['total_mentions'] > 5]
        }

        return market_overview

    def get_current_sentiment_summary(self, symbol: str) -> Dict:
        """Get comprehensive current sentiment summary for a symbol"""
        try:
            conn = sqlite3.connect(self.db_manager.db_path)

            # Get latest aggregated score
            query = '''
                SELECT * FROM aggregated_scores
                WHERE symbol = ?
                ORDER BY timestamp DESC
                LIMIT 1
            '''

            df = pd.read_sql_query(query, conn, params=(symbol,))

            if df.empty:
                return {'symbol': symbol, 'status': 'NO_DATA'}

            latest_data = df.iloc[0].to_dict()
            company_info = self.db_manager.get_company_info(symbol)

            # Get recent sentiment trend
            trend_query = '''
                SELECT timestamp, confidence_weighted_score
                FROM aggregated_scores
                WHERE symbol = ?
                ORDER BY timestamp DESC
                LIMIT 5
            '''

            trend_df = pd.read_sql_query(trend_query, conn, params=(symbol,))
            conn.close()

            # Calculate trend direction
            trend_direction = "STABLE"
            if len(trend_df) >= 2:
                recent_avg = trend_df.head(2)['confidence_weighted_score'].mean()
                older_avg = trend_df.tail(2)['confidence_weighted_score'].mean()

                if recent_avg < older_avg - 0.05:  # Lower score = more positive
                    trend_direction = "IMPROVING"
                elif recent_avg > older_avg + 0.05:
                    trend_direction = "DECLINING"

            summary = {
                'symbol': symbol,
                'company_name': company_info['company_name'],
                'sector': company_info['sector'],
                'current_sentiment_score': latest_data.get('confidence_weighted_score', 0.5),
                'sentiment_status': self.get_sentiment_status(latest_data.get('confidence_weighted_score', 0.5)),
                'trend_direction': trend_direction,
                'total_mentions': latest_data.get('total_mentions', 0),
                'prediction_confidence': latest_data.get('prediction_confidence', 0.0),
                'market_momentum': latest_data.get('market_momentum', 0.5),
                'volatility_risk': latest_data.get('volatility_score', 0.0),
                'last_updated': latest_data.get('timestamp', 'Unknown'),
                'trending_topics': json.loads(latest_data.get('trending_topics', '[]')),
                'regulatory_impact': latest_data.get('regulatory_impact', 0.5)
            }

            return summary

        except Exception as e:
            logging.error(f"Error getting sentiment summary for {symbol}: {e}")
            return {'symbol': symbol, 'status': 'ERROR', 'error': str(e)}

    def start_automated_monitoring(self):
        """Start fully automated monitoring system with enhanced scheduling"""
        def run_collection_loop():
            """Background thread for data collection"""
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)

            while self.is_running:
                try:
                    cycle_results = loop.run_until_complete(self.run_full_collection_cycle())

                    # Log cycle performance
                    if cycle_results['total_records'] == 0:
                        logging.warning("No records collected in this cycle - check data sources")

                    # Wait for next collection cycle
                    time.sleep(self.collection_interval * 60)

                except Exception as e:
                    logging.error(f"Error in collection loop: {e}")
                    time.sleep(60)  # Wait 1 minute before retrying

        def run_aggregation_loop():
            """Background thread for sentiment aggregation"""
            while self.is_running:
                try:
                    aggregation_results = self.calculate_and_save_all_aggregations()

                    # Log market overview
                    market_overview = aggregation_results.get('market_overview', {})
                    if market_overview and market_overview.get('status') != 'NO_DATA':
                        logging.info(f"Market Overview: {market_overview.get('market_status', 'Unknown')} | Active Symbols: {len(aggregation_results.get('symbols_processed', []))}")

                    # Wait for next aggregation cycle
                    time.sleep(self.aggregation_interval * 60)

                except Exception as e:
                    logging.error(f"Error in aggregation loop: {e}")
                    time.sleep(300)  # Wait 5 minutes before retrying

        if self.is_running:
            logging.warning("Monitoring is already running!")
            return

        self.is_running = True

        # Start background threads
        collection_thread = threading.Thread(target=run_collection_loop, daemon=True, name="CollectionThread")
        aggregation_thread = threading.Thread(target=run_aggregation_loop, daemon=True, name="AggregationThread")

        collection_thread.start()
        aggregation_thread.start()

        logging.info("=" * 80)
        logging.info("KENYAN MARKET SENTIMENT MONITORING SYSTEM STARTED")
        logging.info(f"Collection Interval: {self.collection_interval} minutes")
        logging.info(f"Aggregation Interval: {self.aggregation_interval} minutes")
        logging.info(f"Monitoring Symbols: {', '.join(self.symbols)}")
        logging.info("=" * 80)

    def stop_monitoring(self):
        """Stop automated monitoring system"""
        if not self.is_running:
            logging.warning("Monitoring is not currently running!")
            return

        self.is_running = False

        # Close web scraper resources
        self.web_scraper.close_selenium()

        logging.info("KENYAN MARKET SENTIMENT MONITORING STOPPED")
        logging.info(f"Total Collection Cycles: {self.collection_stats['total_cycles']}")
        logging.info(f"Total Records Collected: {self.collection_stats['total_records']}")
        logging.info(f"Success Rate: {((self.collection_stats['total_cycles'] - self.collection_stats['failed_cycles']) / max(self.collection_stats['total_cycles'], 1)) * 100:.1f}%")

    def get_comprehensive_dashboard(self) -> Dict:
        """Get comprehensive dashboard data for all symbols"""
        dashboard = {
            'timestamp': datetime.now(),
            'system_status': {
                'is_running': self.is_running,
                'collection_stats': self.collection_stats
            },
            'symbols': {},
            'market_summary': {}
        }

        # Get data for each symbol
        symbol_summaries = []
        for symbol in self.symbols:
            symbol_data = self.get_current_sentiment_summary(symbol)
            dashboard['symbols'][symbol] = symbol_data
            if symbol_data.get('status') not in ['NO_DATA', 'ERROR']:
                symbol_summaries.append(symbol_data)

        # Generate market summary
        if symbol_summaries:
            dashboard['market_summary'] = self.generate_market_overview(symbol_summaries)

        return dashboard

    def export_sentiment_data(self, symbol: str = None, days: int = 7, format: str = 'csv') -> str:
        """Export sentiment data for analysis"""
        try:
            conn = sqlite3.connect(self.db_manager.db_path)

            if symbol:
                query = '''
                    SELECT * FROM sentiment_data
                    WHERE symbol = ? AND timestamp > datetime('now', '-{} days')
                    ORDER BY timestamp DESC
                '''.format(days)
                df = pd.read_sql_query(query, conn, params=(symbol,))
                filename = f"{symbol}_sentiment_data_{days}days.{format}"
            else:
                query = '''
                    SELECT * FROM sentiment_data
                    WHERE timestamp > datetime('now', '-{} days')
                    ORDER BY timestamp DESC
                '''.format(days)
                df = pd.read_sql_query(query, conn)
                filename = f"all_symbols_sentiment_data_{days}days.{format}"

            conn.close()

            if format.lower() == 'csv':
                df.to_csv(filename, index=False)
            elif format.lower() == 'json':
                df.to_json(filename, orient='records', indent=2)
            else:
                raise ValueError("Format must be 'csv' or 'json'")

            logging.info(f"Exported {len(df)} records to {filename}")
            return filename

        except Exception as e:
            logging.error(f"Error exporting data: {e}")
            return ""

# Demo and Testing Functions
async def run_kenyan_market_demo():
    """Comprehensive demo of the Kenyan market sentiment system"""

    print("=" * 80)
    print("KENYAN STOCK MARKET SENTIMENT ANALYSIS SYSTEM")
    print("=" * 80)
    print("Specialized for Kenyan financial sources and market context")
    print()

    # Initialize with key Kenyan stocks
    kenyan_symbols = [
        'SCOM',    # Safaricom - Telecommunications giant
        'EQTY',    # Equity Group - Major bank
        'KCB',     # KCB Group - Major bank
        'EABL',    # East African Breweries - Consumer goods
        'KPLC',    # Kenya Power - Utilities
        'BRIT',    # Britam - Insurance
        'COOP',    # Co-operative Bank
        'BAT'      # British American Tobacco
    ]

    print(f"Monitoring Symbols: {', '.join(kenyan_symbols)}")
    print("Sentiment Scale: 0.0-0.5 = POSITIVE, 0.5-1.0 = NEGATIVE")
    print()

    # Initialize system
    system = KenyanStockSentimentSystem(
        symbols=kenyan_symbols,
        collection_interval=30,  # 30 minutes between collections
        aggregation_interval=180  # 3 hours between aggregations
    )

    try:
        print("1. RUNNING INITIAL DATA COLLECTION...")
        print("-" * 50)

        # Run initial collection cycle
        initial_results = await system.run_full_collection_cycle()

        print(f"✓ Collection completed in {initial_results['performance_metrics']['duration_seconds']:.1f} seconds")
        print(f"✓ Total records collected: {initial_results['total_records']}")
        print(f"✓ Active sources: {initial_results['performance_metrics']['active_sources']}")
        print()

        print("2. CALCULATING COMPREHENSIVE SENTIMENT AGGREGATIONS...")
        print("-" * 50)

        # Calculate aggregations
        aggregation_results = system.calculate_and_save_all_aggregations()

        market_summary = aggregation_results.get('aggregation_summary', {})
        print(f"✓ Symbols processed: {market_summary.get('total_symbols', 0)}")
        print(f"✓ Total mentions analyzed: {market_summary.get('total_mentions', 0)}")
        print(f"✓ High confidence signals: {market_summary.get('high_confidence_signals', 0)}")
        print()

        print("3. CURRENT SENTIMENT ANALYSIS RESULTS")
        print("-" * 50)

        # Display results for each symbol
        for symbol in kenyan_symbols:
            summary = system.get_current_sentiment_summary(symbol)

            if summary.get('status') not in ['NO_DATA', 'ERROR']:
                status = summary['sentiment_status']
                score = summary['current_sentiment_score']
                mentions = summary['total_mentions']
                confidence = summary['prediction_confidence']
                trend = summary['trend_direction']

                print(f"{symbol:6} | {summary['company_name'][:25]:25} | {status:13} | Score: {score:.3f} | Mentions: {mentions:3} | Confidence: {confidence:.2f} | Trend: {trend}")
            else:
                print(f"{symbol:6} | {summary.get('status', 'UNKNOWN'):50}")

        print()

        # Market overview
        market_overview = aggregation_results.get('market_overview', {})
        if market_overview and market_overview.get('status') != 'NO_DATA':
            print("4. MARKET OVERVIEW")
            print("-" * 50)

            overall_status = market_overview.get('market_status', 'Unknown')
            print(f"Overall Market Sentiment: {overall_status}")

            # Sector analysis
            sector_analysis = market_overview.get('sector_analysis', {})
            if sector_analysis:
                print("\nSector Analysis:")
                for sector, data in sector_analysis.items():
                    print(f"  {sector:15}: {data['status']:13} ({data['companies_count']} companies)")

            # Top performers
            top_performers = market_overview.get('top_performers', [])
            if top_performers:
                print(f"\nTop Performers (Most Positive):")
                for i, perf in enumerate(top_performers[:3], 1):
                    print(f"  {i}. {perf['symbol']} - {perf['sentiment_status']}")

            print()

        print("5. SYSTEM CAPABILITIES")
        print("-" * 50)
        print("✓ Real-time monitoring of 12+ Kenyan financial sources")
        print("✓ Advanced NLP with Kenyan market context")
        print("✓ Predictive confidence scoring")
        print("✓ Sector-specific sentiment analysis")
        print("✓ Regulatory impact assessment")
        print("✓ Automated data collection every 30 minutes")
        print("✓ Sentiment aggregation every 3 hours")
        print("✓ Export capabilities (CSV/JSON)")
        print()

        print("6. USAGE INSTRUCTIONS")
        print("-" * 50)
        print("• Start automated monitoring: system.start_automated_monitoring()")
        print("• Stop monitoring: system.stop_monitoring()")
        print("• Get dashboard: system.get_comprehensive_dashboard()")
        print("• Export data: system.export_sentiment_data('SCOM', days=7)")
        print("• Get symbol summary: system.get_current_sentiment_summary('SCOM')")
        print()

        print("=" * 80)
        print("KENYAN MARKET SENTIMENT SYSTEM READY!")
        print("The system is now configured and ready for automated monitoring.")
        print("All sentiment data will be stored in 'kenyan_market_sentiment.db'")
        print("=" * 80)

        return system

    except Exception as e:
        logging.error(f"Demo error: {e}")
        print(f"❌ Demo encountered an error: {e}")
        return None

if __name__ == "__main__":
    # Run the comprehensive Kenyan market demo
    asyncio.run(run_kenyan_market_demo())

KENYAN STOCK MARKET SENTIMENT ANALYSIS SYSTEM
Specialized for Kenyan financial sources and market context

Monitoring Symbols: SCOM, EQTY, KCB, EABL, KPLC, BRIT, COOP, BAT
Sentiment Scale: 0.0-0.5 = POSITIVE, 0.5-1.0 = NEGATIVE

1. RUNNING INITIAL DATA COLLECTION...
--------------------------------------------------


In [ ]:
!pip install selenium


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.2/499.2 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: typing_extensions
    Found existing installation: typing_extensions 4.15.0
    Uninstalling typing_extensions-4.15.0:
      Successfully uninstalled typing_extensions-4.15.0


In [ ]:
!pip install schedule

In [ ]:
!pip install nest_asyncio

/usr/lib/python3.12/pathlib.py:404: RuntimeWarning: coroutine 'main' was never awaited
  parsed = [sys.intern(str(x)) for x in rel.split(sep) if x and x != '.']


In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
# Install necessary libraries and chrome driver
!apt-get update
!apt-get install -y chromium-browser chromium-chromedriver

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [346 B]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,006 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,274 kB]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,310 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jam